# Generator's LAB

Podstawą wszystkich rozwiązań typu RAG (ang. Retrieval Augmented Generation) jest komponent wyszukiwania. Wyszukiwanie polega na identyfikacji odpowiednich dokumentów, w naszym RAGOwym przypadku takich dokumentów, które potencjalnie zawierają informacje kluczowe do odpowiedzenia na postawione pytanie. Definicja dokumentu jest też rzeczą płynną, raz to może być paragraf, a raz strona, czy rozdział, zależy to od zadania, oraz modeli jakie używamy.

Po realizacji procesu wyszukania top N dokumentów (np. N=100), bardzo często dokonuje się re-ranking aby wybrać top5-10 najlepszych i takie wyselekcjonowane dokumenty tworzą tzw. kontekst. Kontekst to konkatenecja tych top5-10 najlepszych dokumentów. Mówiąc konkatentacja mam na myśli wylistowanie tych dokumentów w formie jednego tekstu.

Na wykładzie starałem się wyraźnie zwrócić uwagę na to, że taki kontekst staje się kluczowym elementem prompta wysyłanego do modelu LLM pełniącego rolę generatora. Generator to LLM pełniący rolę ostatniegu komponentu w RAG architekturze oraz produkującego finalną odpowiedź w języku naturalnym. Generator to model LLM, który po otrzymaniu na wejściu, w prompcie, pytanie i kontekst generuje odpowiedź w języku naturalnym na zadane pytanie w dostarczonym ujętym kontekście informacyjnym. Reasumując, prompt do generatora zawiera pytanie i kontekst, a completion to odpowiedź generatora.

Przykładowey szablon prompta do generatora, który posiada listę dokumentów budujących kontekst informacyjny, oraz instrukcje, a na końcu włąsciwe pytanie.


```
Numerowana lista dokumentów jest poniżej:
---------------------
<results>{% for doc in docs %}
Dokument: {{ loop.index0 }}
{{ doc }}
{% endfor %}</results>
---------------------
Odpowiedz na pytanie użytkownika wykorzystując tylko informacje znajdujące się w dokumentach, a nie wcześniejszą wiedzę.
Udziel wysokiej jakości, poprawnej gramatycznie odpowiedzi w języku polskim. Odpowiedź powinna zawierać cytowania do dokumentów, z których pochodzą informacje. Zacytuj dokument za pomocą symbolu [nr_dokumentu] powołując się na fragment np. [0] dla fragmentu z dokumentu 0. Jeżeli w dokumentach nie ma informacji potrzebnych do odpowiedzi na pytanie, zamiast odpowiedzi zwróć tekst: "Nie udało mi się odnaleźć odpowiedzi na pytanie".

Pytanie: {{ question }}

```


Do pracy wymagamy zainstalowania pakietów - `transformers` oraz `datasets`


In [1]:
!pip install -U transformers datasets bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


Zanim przejdziemy do zdefiniowania szczegółowych zadań sprawdźmy jak użyć powyższy szablon prompta RAGowego do istniejącego LLMa, który ma kompetencje bycia generatorem w RAGach.


Przykładowe użycie generatora na statycznych danych z wykorzystaniem pipeline z transformers oraz modeli PLLuM LLama 8b

In [2]:
# Prompt do generatora z kontekstem opartym na przykładowych wyszukanych tekstach
prepared_prompt = """
Numerowana lista dokumentów jest poniżej:
---------------------

Dokument: 0

Uzyskaj paszport - usługa dla osoby dorosłej
Co musisz przygotować
twoje jedno aktualne, kolorowe zdjęcie - zrób je u fotografa nie wcześniej niż 6 miesięcy przed złożeniem wniosku. Jeśli chcesz je zrobić samodzielnie - sprawdź, jak wykonać zdjęcie do paszportu,
dowód opłaty za paszport (po złożeniu wniosku opłata nie podlega zwrotowi),
jeśli przysługuje ci prawo do zniżki lub zwolnienia z opłaty - dokument, który to potwierdzi, na przykład ważna legitymacja studencka (szczegóły znajdziesz w sekcji Zwolnienia i zniżki),
ważny paszport, którego obecnie używasz. Jeśli nie masz ważnego paszportu - weź ze sobą ważny dowód osobisty,
jeśli twoje nazwisko zostało zmienione po ślubie za granicą - skrócony lub zupełny odpis polskiego aktu małżeństwa.


Dokument: 1

Uzyskaj paszport - usługa dla osoby dorosłej
Jak odbierzesz dokument

Osobiście w punkcie paszportowym - tam, gdzie złożysz wniosek.

Weź ze sobą:

ważny dokument tożsamości,
aktualny dokument paszportowy - jeśli masz,
poprzedni paszport - jeśli masz. Zostanie on anulowany.
Przy odbiorze paszportu sprawdź - czy twoje dane osobowe i biometryczne są poprawne.


---------------------
Odpowiedz na pytanie użytkownika wykorzystując tylko informacje znajdujące się w dokumentach, a nie wcześniejszą wiedzę.
Udziel wysokiej jakości, poprawnej gramatycznie odpowiedzi w języku polskim. Odpowiedź powinna zawierać cytowania do dokumentów, z których pochodzą informacje. Zacytuj dokument za pomocą symbolu [nr_dokumentu] powołując się na fragment np. [0] dla fragmentu z dokumentu 0. Jeżeli w dokumentach nie ma informacji potrzebnych do odpowiedzi na pytanie, zamiast odpowiedzi zwróć tekst: "Nie udało mi się odnaleźć odpowiedzi na pytanie".

Pytanie: Jak odebrać paszport?"""


In [3]:
import transformers
import torch

In [ ]:
model_name = "CYFRAGOVPL/Llama-PLLuM-8B-instruct"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_name,
    model_kwargs={"torch_dtype": torch.float16}, #float16 if not ampere/hopper gpu
    device_map="auto",
)

messages = [
    {"role": "system", "content": "You are an administration chatbot assistant!"},
    {"role": "user", "content": prepared_prompt},
]

outputs = pipeline(
    messages,
    max_new_tokens=256,
)
print(outputs[0]["generated_text"][-1])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/525 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'role': 'assistant', 'content': 'Aby odebrać paszport, należy udać się osobiście do punktu paszportowego, w którym złożono wniosek [1]. Przy odbiorze paszportu należy posiadać ważny dokument tożsamości, a także aktualny dokument paszportowy, jeśli posiada się go [1]. Ponadto, należy mieć przy sobie poprzedni paszport, który zostanie anulowany [1]. Przy odbiorze paszportu należy sprawdzić, czy dane osobowe i biometryczne są poprawne [1].'}


Powyżej pokazałem jak użyć modelu PLLuM LLama 8b Instruct (CYFRAGOVPL/Llama-PLLuM-8B-instruct) do generacji na bazie prompta zawierającego kontekst (lista dokumentów) i pytanie. Teraz czas na twoje zadania.

In [ ]:
import gc


del pipeline
del messages

gc.collect()

9610

ZADANIE #1 - Sprawdź jak działa kod wyżej umieszczony na modelu typu chat - CYFRAGOVPL/Llama-PLLuM-8B-chat, czy widzisz różnice, oraz opisz co zaobserwowałeś jak device_map zmienisz z "auto" na "cuda", wytłumacz dlaczego tak się dzieje?

In [ ]:
model_name = "CYFRAGOVPL/Llama-PLLuM-8B-chat"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_name,
    model_kwargs={"torch_dtype": torch.float16}, #float16 if not ampere/hopper gpu
    device_map="auto",
)

messages = [
    {"role": "system", "content": "You are an administration chatbot assistant!"},
    {"role": "user", "content": prepared_prompt},
]

outputs = pipeline(
    messages,
    max_new_tokens=256,
)
print(outputs[0]["generated_text"][-1])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/931 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/525 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'role': 'assistant', 'content': 'Aby odebrać paszport, należy udać się do punktu paszportowego, w którym złożono wniosek [1]. Przy odbiorze paszportu należy mieć przy sobie ważny dokument tożsamości, aktualny dokument paszportowy - jeśli jest taki - oraz poprzedni paszport, który zostanie anulowany [1]. Przy odbiorze należy również sprawdzić, czy dane osobowe i biometryczne są poprawne [1].'}


- __według dokumentacji model typu 'instructions' został wytrenowany tak aby aby ściśle podążać za przekazanymi instrukcjami i udzielać zwięzłych treściwych odpowiedzi, natomiast model typu 'chat' został przygotowany tak aby prowadzić bardziej naturalną rozmowę/konwersację - widać to również po wygenerowanych odpowiedziach model typu 'instractions' przygotował odpowiedź składającą się z kilku krótkich zdań będących 'parafrazami' kontekstu dokumentu 1, natomiast model typu 'chat' wygenerował bardziej naturalną, podobną do ludzkiej odpowiedź, brzmiącą jak naturalna część rozmowy__

In [ ]:
import gc
import torch

del pipeline
del messages

gc.collect()
torch.cuda.empty_cache()

In [ ]:
model_name = "CYFRAGOVPL/Llama-PLLuM-8B-chat"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_name,
    model_kwargs={"torch_dtype": torch.float16}, #float16 if not ampere/hopper gpu
    device_map="cuda",
)

messages = [
    {"role": "system", "content": "You are an administration chatbot assistant!"},
    {"role": "user", "content": prepared_prompt},
]

outputs = pipeline(
    messages,
    max_new_tokens=256,
)
print(outputs[0]["generated_text"][-1])

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

ValueError: Could not load model CYFRAGOVPL/Llama-PLLuM-8B-chat with any of the following classes: (<class 'transformers.models.auto.modeling_auto.AutoModelForCausalLM'>, <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>). See the original errors:

while loading with AutoModelForCausalLM, an error is thrown:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/pipelines/base.py", line 232, in load_model
    model = model_class.from_pretrained(model, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/auto_factory.py", line 372, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 4109, in from_pretrained
    load_info = cls._load_pretrained_model(model, state_dict, checkpoint_files, load_config)
              

- __ustawienie device_map="cuda" wymusza zaalokowanie całego modelu do pamięci VRAM aktualnie używanej karty graficznej, użyty model wymaga dostępności ok. większej ilości VRAMu niż dostępna jest w darmowych zasobach google collab - skutkuje to nie zmieszczeniem się modelu w pamięci i błędem torch.OutOfMemoryError, natomiast device_map="auto" używa biblioteki do dynamicznego sprawdzenia dostępności pamięci oraz dzieli wagi modelu i ładuje je do pamięci tak aby nie przekroczyć dostępnych zasobów - część wag załaduje do VRAMu GPU, natomiast pozostałą część do RAMu CPU__

In [ ]:
import gc
import torch


gc.collect()
torch.cuda.empty_cache()

ZADANIE #2 - Przepisz wyżej pokazany kod tak aby nie używać pipelinu, tylko natywnego podejścia transformers z użyciem takich komend jak AutoModelForCausalLM.from_pretrained(), AutoTokenizer.from_pretrained() i tokenizer.apply_chat_template(), oraz model.generate()

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
messages = [
    {"role": "system", "content": "You are an administration chatbot assistant!"},
    {"role": "user", "content": prepared_prompt},
]

base_model_id = "CYFRAGOVPL/Llama-PLLuM-8B-instruct"
q_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=q_config, dtype=torch.float16, device_map="auto", trust_remote_code=True, low_cpu_mem_usage=True)
print(model)

tokenizer = AutoTokenizer.from_pretrained(base_model_id, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

chat_template = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
print(f"Tokenized prompt:\n {tokenizer.decode(chat_template['input_ids'][0])}")

output = model.generate(**chat_template, max_new_tokens=256)
print(tokenizer.decode(output[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/525 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Tokenizer vocab size: 128000
Tokenized prompt:
 <|begin_of_text|>[INST]You are an administration chatbot assistant!


Numerowana lista dokumentów jest poniżej:
---------------------

Dokument: 0

Uzyskaj paszport - usługa dla osoby dorosłej
Co musisz przygotować
twoje jedno aktualne, kolorowe zdjęcie - zrób je u fotografa nie wcześniej niż 6 miesięcy przed złożeniem wniosku. Jeśli chcesz je zrobić samodzielnie - sprawdź, jak wykonać zdjęcie do paszportu,
dowód opłaty za paszport (po złożeniu wniosku opłata nie podlega zwrotowi),
jeśli przysługuje ci prawo do zniżki lub zwolnienia z opłaty - dokument, który to potwierdzi, na przykład ważna legitymacja studencka (szczegóły znajdziesz w sekcji Zwolnienia i zniżki),
ważny paszport, którego obecnie używasz. Jeśli nie masz ważnego paszportu - weź ze sobą ważny dowód osobisty,
jeśli twoje nazwisko zostało zmienione po ślubie za granicą - skrócony lub zupełny odpis polskiego aktu małżeństwa.


Dokument: 1

Uzyskaj paszport - usługa dla osoby d

In [ ]:
import gc
import torch

del model
del tokenizer
del chat_template
del messages

gc.collect()
torch.cuda.empty_cache()

ZADANIE #3 - Do kodu napisanego w zadaniu 2 wprowadź optymalizacje obliczeń atencji np. flash-attention-2, aby zmniejszyć zużycie pamięci i szybkość generacji, opisz co zaobserwowałeś, czy coś się zmieniło i dlaczego nie, albo tak?

In [6]:
messages = [
    {"role": "system", "content": "You are an administration chatbot assistant!"},
    {"role": "user", "content": prepared_prompt},
]

### tested - flash attn not available for T4 architecture
# base_model_id = "CYFRAGOVPL/Llama-PLLuM-8B-instruct"
# q_config = BitsAndBytesConfig(load_in_8bit=True)
# model = AutoModelForCausalLM.from_pretrained(
#     base_model_id,
#     quantization_config=q_config,
#     dtype=torch.float16,
#     attn_implementation="flash_attention_2",  # based on hf example: https://huggingface.co/docs/transformers/v4.48.0/llm_optims#flashattention-2$0
#     trust_remote_code=True,
#     low_cpu_mem_usage=True)

base_model_id = "CYFRAGOVPL/Llama-PLLuM-8B-instruct"
q_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=q_config,
    dtype=torch.float16,
    attn_implementation="sdpa",
    trust_remote_code=True,
    low_cpu_mem_usage=True)


print(model)

tokenizer = AutoTokenizer.from_pretrained(base_model_id, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

chat_template = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
# print(f"Tokenized prompt:\n {tokenizer.decode(chat_template['input_ids'][0])}")

output = model.generate(**chat_template, max_new_tokens=256)
print("Model response: \n")
print(tokenizer.decode(output[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/525 [00:00<?, ?B/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Tokenizer vocab size: 128000


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Model response: 

You are an administration chatbot assistant!


Numerowana lista dokumentów jest poniżej:
---------------------

Dokument: 0

Uzyskaj paszport - usługa dla osoby dorosłej
Co musisz przygotować
twoje jedno aktualne, kolorowe zdjęcie - zrób je u fotografa nie wcześniej niż 6 miesięcy przed złożeniem wniosku. Jeśli chcesz je zrobić samodzielnie - sprawdź, jak wykonać zdjęcie do paszportu,
dowód opłaty za paszport (po złożeniu wniosku opłata nie podlega zwrotowi),
jeśli przysługuje ci prawo do zniżki lub zwolnienia z opłaty - dokument, który to potwierdzi, na przykład ważna legitymacja studencka (szczegóły znajdziesz w sekcji Zwolnienia i zniżki),
ważny paszport, którego obecnie używasz. Jeśli nie masz ważnego paszportu - weź ze sobą ważny dowód osobisty,
jeśli twoje nazwisko zostało zmienione po ślubie za granicą - skrócony lub zupełny odpis polskiego aktu małżeństwa.


Dokument: 1

Uzyskaj paszport - usługa dla osoby dorosłej
Jak odbierzesz dokument

Osobiście w punkcie 

* __przetestowane zostało użycie optymalizacji `flash attention` jednakże próba zbudowania whella paczki kończyła się `wheel build error` - według dokumentacji mechanizm flash attention został przygotowany pod nowsze architektury niż dostępna w darmowym planie google colab GPU T4, dlatego też nie jest on obsługiwany na tej architekturze, zamiast `falsh attention` użyta została optymalizacja `sdpa`, która dynamicznie wybiera najlepszą wersję optymalizacji działającą na używanej architekturze__
* __wniosek: sama generacja (nie wliczając przygotowania modelu, pobierania wag, kwantyzacji itp.) była zauważalnie szybsza od generacji odpowiedzi przez modele testowane wyżej bez zastosowanej optymalizacji atencji modelu, również zasoby architektury T4 były mniej obciążone podczas tej generacji (w porównaniu do poprzednich)__